## Implementation of Decision Tree
---

## Prerequisites
- Decision Tree 
- Familiarity with Object-Oriented Programming in Python
- Comfortable with graph data structures


## Learning Objectives 
After going through this notebook, students should be able to: 
- Find the best split when the dataset has continuous attributes.
- Use object-oriented programming methodology to create a decision tree from scratch.


## Introduction

In the previous notebook, we discussed decision tree algorithms. Here we will implement decision trees from scratch using python and its libraries. This implementation will use a different algorithm than the ones we studied in the earlier chapter. This is to reduce the complexity and for easy understandability. The algorithm that we are going to use is as follows: 



> 1.For all instances in a dataset($D$)
>> If all instances belong to the same class $C$, or other stopping criteria(maximum depth, pure node) are met
>>> Create a leaf node and stop 

>> Else 
>>> Compute gini for all attributes.   
>>> Select an attribute(say $f$) that yields the lowest gini.   
>>> Split the data into subsets according to the value of attribute $f$.   

>2.Apply algorithm recursively from step-1 for each of the subsets. 

You might have already noticed that this algorithm doesn't use any pruning technique and is most similar to ID3. Unlike ID3, which uses entropy to find the best split, this algorithm uses gini as the impurity metric. 

Another important difference is the way we handle numerical attributes. We will be using the approach discussed in the chapter **Impurity metrics**. The steps can be summarized as below: 
1. Compute potential threshold values for each attribute. 
2. Use each threshold value to compute the impurity metric(gini).
3. Repeat steps 1 and 2 for all attributes and threshold values.
4. Choose the threshold and attribute value corresponding to the lowest gini to create the split.

## Imports  
Let's import the necessary libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import random
from pprint import pprint
from sklearn import datasets

## Loading and splitting the dataset

In this notebook, we will use the **iris** dataset from sklearn's [`datasets`](https://scikit-learn.org/stable/datasets/index.html) module. We will create a pandas dataframe using the loaded data so that it is easier to create our own `train_test_split` function.

In [ ]:
iris = datasets.load_iris()

# Convert the data into dataframe
data = pd.DataFrame(data= np.c_[iris['data'], iris['target']],
                     columns= iris['feature_names'] + ['target'])
data.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0.0
1,4.9,3.0,1.4,0.2,0.0
2,4.7,3.2,1.3,0.2,0.0
3,4.6,3.1,1.5,0.2,0.0
4,5.0,3.6,1.4,0.2,0.0


Now let us write a function to split the original dataset into train and test set.

In [ ]:
def train_test_split(df, test_size):

    '''Splits the datset into train and test set.

    Parmeters:
    ---------
    df 
        The dataframe than needs to be split into train and test set.

    test_size
    The size of the test set.

    Returns:
    --------
    train_df
    Data to be used as the train set.

    test_df
    Data to be used as the test set.
    '''
    
    if isinstance(test_size, float):
        '''If test_size is float multiply by the length of dataframe to 
        compute the test size of the dataset.
        '''
        test_size = round(test_size * len(df))

    indices = df.index.tolist()
    test_indices = random.sample(population=indices, k=test_size)

    test_df = df.loc[test_indices]
    train_df = df.drop(test_indices)
    
    return train_df, test_df

Using the above function, let's divide the dataset into train and test set. We will keep 20 instances in the test set. For reproducibility, we will also set the random seed to 0.

In [ ]:
# Exercise
random.seed(0)
train_df, test_df = train_test_split(data, test_size=20)

In [ ]:

train_df.shape, test_df.shape

((130, 5), (20, 5))

`train_df` and `test_df` hold the train and test data, respectively. 

## Helper Functions
---

From the above algorithm for decision tree, it is clear that we will need to implement different functions. The list below represents functions we need to create: 
* `get_potential_splits()` - Function to generate/find the potential splits(threshold values). 
* `compute_gini()` - Function to compute gini for the given data.
* `compute_weighted_gini()` - Function to compute weighted gini for given splits/nodes.
* `split_data()` - Function to split the input data into two halves around the given split value.
* `determine_best_split()` - Function to find the best split from a list of potential splits.

Let us begin with the first function `get_potential_splits()`.

In [ ]:
def get_potential_splits(data):
    '''Finds out potential split values for all feature
    columns of the datset. 
    
    Parameter:
    ----------
    data:
        Train set.

    Returns:
    --------
    potential_splits
        A dictionary column name as key and array of potential 
        split values as value.
    '''
    
    potential_splits = {}
    _, n_columns = data.shape
    for column_index in range(n_columns - 1): # excluding the last column which is the label
        potential_splits[column_index] = []
        values = data[:, column_index]
        unique_values = np.unique(values) # np.unique() returns sorted unique values in ascending order

        for index in range(len(unique_values)):
            if index != 0:
                current_value = unique_values[index]
                previous_value = unique_values[index - 1]
                potential_split = (current_value + previous_value) / 2
                
                potential_splits[column_index].append(potential_split)
        
    return potential_splits

The function below, `compute_gini()`, uses the following formula to compute gini:  
  
$\text{Gini} = 1 - \sum_{i=1}^{c}{P_i^2}$   
Here,   
$c = $ Total number of labels/classes.   
$P_i = $ Probability of an item belonging to class $i$. 

In [ ]:
def compute_gini(data):
    '''Function to compute the value of gini index.

    Parameter:
    ----------
        data  
        Data for which the gini is to be computed.

    Returns:
    --------
        gini 
        Gini index for the given data.
    '''
    label_column = data[:, -1]
    _, counts = np.unique(label_column, return_counts=True)

    probabilities = counts / counts.sum()
    gini  = 1 - sum(probabilities ** 2)
        
    return gini

The function below,`comput_weighted_gini()`, computes the weighted average of gini for the left and right child. 

$\text{weighted gini} = \frac{n_L}{n}*\text{Gini}_\text{left} + \frac{n_R}{n}*\text{Gini}_\text{right}$ 

Where, 
$n_L =$ Number of instances in the left child.   
$n_R =$ Number of instances in the right child.   
$n = $ Total number of instances $= n_L + n_R$.   
$\text{Gini}_\text{left} = $ Gini impurity of the left child.   
$\text{Gini}_\text{right} = $ Gini impurity of the right child.  

In [ ]:
def compute_weighted_gini(data_below, data_above):
    '''Function to compute weighted gini.

    Parameters:
    -----------
        data_below, data_above 
        Two splits or subsets of data.

    Returns:
    --------
        weighted_gini
        Weighted gini of the subsets.
    '''
    n = len(data_below) + len(data_above)
    p_data_below = len(data_below) / n
    p_data_above = len(data_above) / n

    weighted_gini =  (p_data_below * compute_gini(data_below) 
                        + p_data_above * compute_gini(data_above))

    return weighted_gini

In [ ]:
def split_data(data, split_column, split_value):
    '''Function to split the data into two parts. 

    Parameters:
    ----------
    data
        Train set or a subset of train set.
    split_column
        Name of attribute used for creating splits.
    split_value
        Threshold value used to split the data.

    Returns:
    --------
    data_below data_above
        Two splits or subsets. 
    '''
    split_column_values = data[:, split_column]

    data_below = data[split_column_values <= split_value]
    data_above = data[split_column_values > split_value]

    return data_below, data_above

In [ ]:
def determine_best_split(data, potential_splits):
    '''Function to get the best split from an array of potential_splits.
    The best split is the one that yields the lowest/minimum gini.

    Parameter:
    ----------
    data
        Data used to find the best split.
    potential_splits
        An array of potential splits.

    Returns:
    --------
    best_split_column
        Attribute that results in the lowest gini.

    best_split_value
        Threshold value corresponding to the lowest gini.

    '''
    overall_gini = 9999 # Since we need to find lowest gini, we initialize
    # overall_gini to a very high value 
    for column_index in potential_splits:
        for value in potential_splits[column_index]:
            data_below, data_above = split_data(data, split_column=column_index, split_value=value)
            current_overall_gini = compute_weighted_gini(data_below, data_above)

            if current_overall_gini <= overall_gini:
                overall_gini = current_overall_gini
                best_split_column = column_index
                best_split_value = value

    return best_split_column, best_split_value

## Classes 
---

To create and train our decision tree, we will create two classes:
* `Node`: This class represents each node in a decision tree. It also contains functions to check if a node is pure and function to assign a label to a node.
* `DecisionTree`: This class represents the entire decision tree. It contains functions to train the tree and make predictions using the tree. It uses the Node class to arrange nodes in a tree. 

Let us create these classes starting with the `Node` class.

In [ ]:
class Node:
    '''Class to represent Node in a decision tree.

        Attributes
        ----------
        parent 
        Parent node.
        left_child
        Left child of the current node.
        right_child
        Right child of the current node.
        data
        Instances of the dataset.
        gini
        Value of gini impurity for data.
        label
        Label assigned to the node.
        is_leaf
        Represents whether a node is leaf or not.
        split_column
        Attribute/Feature used to create the split.
        split_value
        Threshold value used for splitting.
        question
        String representing the attribute test.
        depth
        Depth of the node.


        Methods
        -------
        is_pure
        Checks if the node is pure or not.
        set_label
        Assigns label to a node. 
    '''
    # Exercise
    def __init__(self, parent=None, left_child=None, right_child=None, data=None, 
                gini=None, label=None, is_leaf=None, split_column=None, 
                split_value=None, question=None, depth=None):
        self.parent = parent # Parent of current node
        self.left_child = left_child # Left child of current node
        self.right_child = right_child # Right child of current node
        self.data = data # Data to be stored in the node
        self.label = label # Label associated with the node
        self.is_leaf = is_leaf # Identifies if a node is a leaf or not
        self.split_column = split_column # Feature to test for the node 
        self.split_value = split_value # Split value for the node
        self.question = question # Question representing the test performed in the node
        self.depth = depth # Depth of the node

    def is_pure(self):
        '''Checks if the data in a node are pure or not.

        Returns:
        --------
        True 
        If all instances in the node have the same label.
        
        False
        If instances in the node have different labels.
        '''
        # Exercise
        target = self.data[:, -1]
        unique_classes = np.unique(target)
        if len(unique_classes) == 1:
            return True
        else:
            return False
    
    def set_label(self):
        '''Assigns a label to a node.
        ''' 
        # Exercise
        target = self.data[:, -1]
        unique_classes, counts_unique_classes = np.unique(target, return_counts=True)

        index = counts_unique_classes.argmax()
        classification = unique_classes[index]
        self.label = classification

In [ ]:
class DecisionTree:
    '''Class to represent Decision tree. 

        Attributes
        ----------
        root 
        Root node of the decision tree.
        features
        List of columns in the dataset.
        max_depth
        Maximum depth of the tree.
        min_samples
        Minimum number of samples required in the node.

        Methods
        -------
        train_tree
        Recursively trains a decision tree.
        predict
        Predicts label for a an instance.
        get_root
        Returns the root node of the decision tree. 
    '''
    # Exercise
    def __init__(self,min_samples=2, max_depth=9999, root=None, features=None):
        self.root = root # Root node of the decision tree
        self.root.depth = 0 # Depth of root, a constant value set to 0
        self.max_depth = max_depth # Maximum allowed depth for the decision tree
        self.features = features # Names of features of the data
        self.min_samples = min_samples # Minimum samples allowed in a node
    
    def train_tree(self, node):
        '''Recursively trains a decision tree.
        
        Parameter:
        ----------
        node
        Root node of the decision tree.
        '''
        # Exercise 
        if node.parent:
            node.depth = node.parent.depth + 1 # Task 1
    
        if node.is_pure() or len(node.data)<=self.min_samples or node.depth == self.max_depth:
            node.is_leaf = True  #Task 2
            node.set_label()

        else: # Task 3
            potential_splits = get_potential_splits(node.data)
            split_column, split_value = determine_best_split(node.data, 
                                                            potential_splits)
            
            node.split_column = split_column
            node.split_value = split_value

            data_below, data_above = split_data(node.data, split_column, 
                                                split_value)
            
            node_left = Node(parent=node, data=data_below)
            node_right = Node(parent=node, data=data_above)

            node.left_child = node_left
            node.right_child = node_right
            node.question = "{} <= {}".format(self.features[split_column], split_value)

            self.train_tree(node_left)
            self.train_tree(node_right)
        
    def predict(self, instance):
        '''Predicts class for given instance.

        Parameter:
        ----------
        instance
        An array representing an instance of the data.

        Returns:
        --------
        label
        Predicted label.
        '''
        # Exercise 
        node = self.root  # task 1
        
        while node.is_leaf != True:
            if instance[node.split_column] <= node.split_value: # Task 2
                node = node.left_child
            else:
                node = node.right_child
        return node.label

    def get_root(self):
        '''
        Returns the root node of the decision tree.

        Returns:
        --------
        root
        root node of the decision tree.
        '''
        return self.root # Exercise

After writing all the required methods and classes, let's initialize the `Node()` object. It will be used as a root node for the decision tree. It takes the entire training set as input through the parameter `data`.

In [ ]:
root = Node(data=train_df.values)

Now, let's create a `DecisionTree()` object. It takes following parameters as input to initialize an object:  
* `min_samples`  
* `max_depth`
* `root`
* `features` 

In [ ]:
features = ['sepal length', 'sepal width', 'petal length', 'petal width']

my_tree = DecisionTree(root=root, features=features)

Now, let's train the decision tree using the `train_tree()` method. The method takes the root node as input. 

In [ ]:
my_tree.train_tree(root)

Let's make a prediction using the decision tree. To make the prediction, we will use the first instance from our test set, i.e., `test_df`.

In [ ]:
sample = test_df.values[1][:4] # Features of sample at index 1
label_test = None # Stores the prediction for the test data
label_test = my_tree.predict(sample) 
print(label_test)

2.0


This is how we can create a decision tree from scratch using python. You should now be able to write code for any decision tree algorithm discussed in the earlier chapter. 

## Key Takeaways

* Given an algorithm for decision tree, we can use python to implement it from scratch. 
* Scikit-learn uses a modified version of CART to create decision tree. If you want to create decision tree based on other algorithms, you can follow the process used in this chapter. 

## Additional Resources

### Decision Tree Implementation:
* Google Developers, [Decision Tree from Scratch](https://www.youtube.com/watch?v=LDRbO9a6XPU)



